# TESLA Cavity

This notebook demonstrates a complete TESLA waveguide analysis workflow:

1. Geometry creation
2. Frequency-domain analysis
3. Model order reduction
4. Comparison with analytical solution

In [1]:
from analytical.cst_result import CSTResult
import matplotlib.pyplot as plt
from core.em_project import EMProject
from geometry.importers import TESLACavity

%matplotlib widget
plt.rcParams['figure.dpi'] = 100

## 1. Define Geometry

Create a rectangular waveguide with specified dimensions and mesh parameters.

In [2]:
# tesla = TESLACavity(r"tesla1cell.geo")
# tesla.save_step(r"./tesla1cell.step")

In [3]:
project_name = 'tesla1cell'
base_dir = r"C:\Users\Soske\Documents\git_projects\cavsim3d_simulations"
proj = EMProject(name=project_name, base_dir=base_dir)

Project 'tesla1cell' exists. Loading automatically...
INFO:: 
INFO:: Structure Topology
INFO:: ============================================================
INFO:: Type: Single structure
INFO:: Domains (1): ['cell_1']
INFO:: Total Ports (2): ['port1', 'port2']
INFO:: 
Domain-Port Mapping:
  cell_1: ['port1 (external, input)', 'port2 (external, output)']
Restored PortEigenmodeSolver with 2 ports, 6 total modes
FrequencyDomainSolver state loaded from C:\Users\Soske\Documents\git_projects\cavsim3d_simulations\tesla1cell\fds
INFO:: 
INFO:: Structure Topology
INFO:: ============================================================
INFO:: Type: Single structure
INFO:: Domains (1): ['cell_1']
INFO:: Total Ports (2): ['port1', 'port2']
INFO:: 
Domain-Port Mapping:
  cell_1: ['port1 (external, input)', 'port2 (external, output)']


In [4]:
# 1. Load and prepare geometry
geo = proj.import_geometry(r"./tesla1cell.igs", auto_build=False)
# geo.add_splitting_plane_at_x(0.0577)
# geo.add_splitting_plane_at_x(-0.0577)
# geo.split()
#
# geo.show_split_preview()
geo.build()
geo.name_solids()
print(type(geo))

geo.generate_mesh() # after naming solids, must generate mesh but avoid rebuilding

RuntimeError: Failed to read STEP file: ./tesla1cell.igs

In [ ]:
geo.show('mesh')
geo.mesh.GetBoundaries()
proj.fds.bc

In [ ]:
fom_config = {
    'nportmodes': 3,
    'order': 1,
    'nsamples': 100,
    'fmin': 1e-3,
    'fmax': 5,
    'solver_type': 'iterative',
    'rerun': True
}
fom_result = proj.fds.solve(config=fom_config)

In [ ]:
proj.fds.plot_port_mode('port1', mode=0, component='abs')

In [ ]:
from analytical.cst_result import CSTResult

cstresult = CSTResult(r'C:\Users\Soske\Documents\CST\tesla1cell')
# plot comparison
which = [['1(1)1(1)'], ['2(1)1(1)']]
fig, axs = plt.subplot_mosaic([[1, 2], [3, 4]], figsize=(8, 8), layout='constrained')
for idx, wh in enumerate(which):
    # plot magnitude
    cstresult.plot_s(wh, ax=axs[idx+1])
    proj.fds.fom.plot_s(wh, ax=axs[idx+1])
    # plot phase
    cstresult.plot_s(wh, plot_type='phase', ax=axs[idx+3])
    proj.fds.fom.plot_s(wh, plot_type='phase', ax=axs[idx+3])

In [ ]:
# Reduce model order
rom = proj.fds.fom.reduce()
# concat = proj.fds.fom.rom.concatenate()
roms_config = {
    'nportmodes': 2,
    'nsamples': 1000, # <- changed for more frequency samples
    'fmin': 1e-3,
    'fmax': 5,
    'solver_type': 'direct' # <- changed to direct method, faster for smaller matrices
}
rom_result = rom.solve(config=roms_config) # solve reduced order model on more frequency samples


In [ ]:
from analytical.cst_result import CSTResult
import matplotlib.pyplot as plt
# compare fom, rom, CST and analytical solution
# get cst solution

cstresult = CSTResult(r'C:\Users\Soske\Documents\CST\tesla1cell')
# plot comparison
which = [['1(3)1(3)'], ['2(3)1(3)']]
fig, axs = plt.subplot_mosaic([[1, 2], [3, 4]], figsize=(8, 8), layout='constrained')
for idx, wh in enumerate(which):
    # plot magnitude
    cstresult.plot_s(wh, ax=axs[idx+1], lw=3)
    rom.plot_s(wh, ax=axs[idx+1])
    # plot phase
    cstresult.plot_s(wh, plot_type='phase', ax=axs[idx+3], lw=3)
    rom.plot_s(wh, plot_type='phase', ax=axs[idx+3])

f

In [ ]:
proj.fds.plot_port_mode('port1', 1)

In [ ]:
proj.fds.plot_field(23, domain='cell_1')

In [ ]:
fig, ax = proj.fds.foms.plot_residual(figsize=(7, 7))

In [ ]:
# fig, ax = plot_eigenfrequencies([fds, roms_concat], analytical=cstresult,
#                       labels=['FOM', 'ROM'], n_modes=50)
# ax.set_ylim(0.1, 5)
# ax.set_xlim(0.1, 5)
# plt.show()

In [ ]:
# %%time

# from netgen.occ import *
# from ngsolve import *
# from ngsolve.webgui import Draw
# from ngsolve.bem import *

# kappa=10
# order=4

# screen = WorkPlane(Axes( (0,0,0), Z, X)).RectangleC(15,15).Face()

# sp = Fuse(Sphere( (0,0,0), pi).faces)
# screen.faces.name="screen"
# sp.faces.name="sphere"
# shape = Compound([screen,sp])

# mesh = shape.GenerateMesh(maxh=5/kappa).Curve(order)
# Draw (mesh);

# fes_sphere = Compress(SurfaceL2(mesh, order=order, complex=True, definedon=mesh.Boundaries("sphere")))
# u,v = fes_sphere.TnT()
# fes_screen = Compress(SurfaceL2(mesh, order=order, dual_mapping=True, complex=True, definedon=mesh.Boundaries("screen")))
# print ("ndof_sphere = ", fes_sphere.ndof, "ndof_screen =", fes_screen.ndof)

# with TaskManager():
#     # V = HelmholtzSingleLayerPotentialOperator(fes_sphere, fes_sphere, kappa=kappa, intorder=10)
#     # K = HelmholtzDoubleLayerPotentialOperator(fes_sphere, fes_sphere, kappa=kappa, intorder=10)
#     # C = HelmholtzCombinedFieldOperator(fes_sphere, fes_sphere, kappa=kappa, intorder=10)
#     C = HelmholtzCF(u*ds("sphere"), kappa)*v*ds
#     u,v  = fes_sphere.TnT()
#     Id = BilinearForm(u*v*ds).Assemble()

# lhs = 0.5 * Id.mat + C.mat
# source = exp(1j * kappa * x)
# rhs = LinearForm(-source*v*ds).Assemble()

# gfu = GridFunction(fes_sphere)
# pre = BilinearForm(u*v*ds, diagonal=True).Assemble().mat.Inverse()
# with TaskManager():
#     gfu.vec[:] = solvers.GMRes(A=lhs, b=rhs.vec, pre=pre, maxsteps=40, tol=1e-8)

In [ ]:
# project_name = fr'C:\Users\Soske\Documents\git_projects\cavsim3d_simulations\tesla3cell'
# fds = FrequencyDomainSolver.load(project_name)

In [ ]:
# # 1. Load and prepare geometry
# midcell = STEPImporter(r"./tesla_midcell.step", auto_build=False)
# endcell_l = STEPImporter(r"./tesla_endcell_l.step", auto_build=False)
# endcell_r = STEPImporter(r"./tesla_endcell_r.step", auto_build=False)
# midcell.build()
# endcell_l.build()
# endcell_r.build()

# tesla = Assembly(main_axis='X')
# tesla.add('endcell_l', endcell_l)
# tesla.add('midcell1', midcell, after="endcell_l")
# tesla.add('endcell_r', endcell_r, after='midcell1')

# tesla.inspect()
# tesla.build()
# # geo.name_solids()
# tesla.generate_mesh(maxh=0.01)
# # tesla.mesh.GetBoundaries()